#Game Classification Model
----------------------------
- Now that we have  curated (gold layer data) we want to predict the results of games.
- Games can be classified in many ways
- Home Win , Away Loss, (HWAL)
- Home loss , Away Win (AWHL)
- Draw (D)
- We will be predicting the home_or_away_winner field.

# Import Necessarry Libraries
----------------------------------

In [0]:
### Initially we will use a simple logistical regression to classify game results.
import numpy as np
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import mlflow

#Parameters
-----------------

In [0]:
catalog = 'wc_2026_predions'
gold_schema = 'gold'
gold_table_name = 'internation_results_data'
logstic_regression_table_name = 'model_pred_data'

# Read in the Gold Layer Data
--------------------------------

In [0]:
from pyspark.sql.functions import col

df_gold = spark.read.table(f'{catalog}.{gold_schema}.{gold_table_name}')

display(df_gold.where(col('home_score').isNull()))


#Seperate Testing and Preditions
----------------------------

In [0]:
from pyspark.sql.functions import col, current_date
from datetime import datetime

# Identify future games (games that haven't been played yet)
# Future games are identified by:
#   1. date > today (games scheduled for the future)
#   2. OR home_score/away_score are null (no result recorded)

print("Separating historical and future games...\n")

# Get current date
today = current_date()

# Separate future games
df_future = df_gold.filter( 
    (col('home_score').isNull()) | 
    (col('away_score').isNull())
)

# Keep only historical games for training
df_historical = df_gold.filter(
    (col('home_score').isNotNull()) & 
    (col('away_score').isNotNull())
)

future_count = df_future.count()
historical_count = df_historical.count()
total_count = df_gold.count()

print(f"Total games: {total_count:,}")
print(f"Historical games (for training): {historical_count:,} ({historical_count/total_count*100:.1f}%)")
print(f"Future games (for testing): {future_count:,} ({future_count/total_count*100:.1f}%)")

if future_count > 0:
    print(f"\n✓ Found {future_count:,} future games to test on after training!")
    print("\nSample of future games:")
    display(df_future.select('date', 'home_team', 'away_team', 'tournament', 'home_score', 'away_score').limit(10))
else:
    print("\n⚠️ No future games found. All games have been played.")

# Use historical games for model training
df_gold = df_historical
print(f"\nProceeding with {df_gold.count():,} historical games for model training...")

In [0]:
# Add label encoding to the DataFrame
# Map home_or_away_winner to numeric labels: draw=0, home=1, away=2
from pyspark.sql.functions import when

df_gold = df_gold.withColumn(
    'label',
    when(col('home_or_away_winner') == 'draw', 0)
    .when(col('home_or_away_winner') == 'home', 1)
    .when(col('home_or_away_winner') == 'away', 2)
    .otherwise(None)
)

print("✓ Label column created")
print("  0 = draw")
print("  1 = home win")
print("  2 = away win")

# Show label distribution
df_gold.groupBy('label', 'home_or_away_winner').count().orderBy('label').show()

In [0]:
# Convert Spark DataFrame to pandas for scikit-learn ML pipeline
# For ~10K rows, pandas/sklearn is faster and has no cache limits
print("Converting Spark DataFrame to pandas...")
df_pandas = df_gold.toPandas()

print(f"✓ Conversion complete!")
print(f"Shape: {df_pandas.shape}")
print(f"Memory usage: {df_pandas.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df_pandas.head()

In [0]:
# Import scikit-learn libraries for ML pipeline
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    LabelEncoder,
    OneHotEncoder,
    OrdinalEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd

print("✓ Scikit-learn libraries imported")

In [0]:
# Define feature columns and target variable
# Target: 'label' (0=draw, 1=home win, 2=away win)
y = df_pandas['label']

# Drop columns we don't want as features
X = df_pandas.drop(columns=['label', 'home_or_away_winner'])

print(f"✓ Features and target separated")
print(f"\nFeature columns ({len(X.columns)}):")
print(X.columns.tolist())
print(f"\nTarget distribution:")
print(y.value_counts().sort_index())
print(f"  0 = draw, 1 = home win, 2 = away win")

In [0]:
# Define column groups for different preprocessing strategies

# Numerical columns: need imputation and scaling
# REMOVED OVERFITTING FEATURES:
#   - home_result_score, away_result_score (direct outcome encoding - data leakage!)
#   - form features (derived from recent game outcomes)
numerical_features = [
    'home_avg_goals', 'home_game_number',
    'away_avg_goals', 'away_game_number'
]

# Categorical columns (low/medium cardinality): use ordinal encoding
categorical_features = ['city', 'country', 'tournament']

# Team columns (high cardinality): use one-hot encoding
# Now we CAN use OneHotEncoder without memory issues!
team_features = ['home_team', 'away_team']

# Boolean column: pass through as-is
boolean_features = ['neutral']

print(f"Feature groups defined:")
print(f"  Numerical: {len(numerical_features)} columns")
print(f"  Categorical: {len(categorical_features)} columns")
print(f"  Teams: {len(team_features)} columns")
print(f"  Boolean: {len(boolean_features)} column")

# Build preprocessing pipeline using ColumnTransformer
# This applies different transformations to different column types
preprocessor = ColumnTransformer(
    transformers=[
        # Numerical: impute with median, then standardize
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numerical_features),
        
        # Categorical: encode as ordinal (preserves order, memory efficient)
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), 
         categorical_features),
        
        # Teams: one-hot encode (creates binary columns for each team)
        # handle_unknown='ignore' means unseen teams in test data won't cause errors
        ('team', OneHotEncoder(handle_unknown='ignore', sparse_output=False), 
         team_features),
        
        # Boolean: pass through without transformation
        ('bool', 'passthrough', boolean_features)
    ],
    verbose_feature_names_out=False  # Simpler feature names in output
)

print(f"\n✓ Preprocessing pipeline constructed!")
print(f"  - Handles missing values (median imputation)")
print(f"  - Standardizes numerical features (mean=0, std=1)")
print(f"  - Encodes categorical variables")
print(f"  - One-hot encodes teams (~300+ binary features)")
print(f"  - Total stages: {len(preprocessor.transformers)}")

In [0]:
# Fit the preprocessor on the data and transform it
# This learns all parameters (means, stds, encodings) and applies transformations
print("Fitting and transforming data...")
X_transformed = preprocessor.fit_transform(X)

# Get feature names after transformation
feature_names = preprocessor.get_feature_names_out()

print(f"\n✓ Transformation complete!")
print(f"\nDataset shape:")
print(f"  Original features: {X.shape}")
print(f"  Transformed features: {X_transformed.shape}")
print(f"  Target: {y.shape}")
print(f"\nFeature expansion:")
print(f"  Started with {X.shape[1]} columns")
print(f"  Ended with {X_transformed.shape[1]} features")
print(f"  Expansion due to one-hot encoding teams (~300+ teams → ~600+ binary features)")

# Show sample of feature names
print(f"\nSample of feature names (first 20):")
for i, name in enumerate(feature_names[:20], 1):
    print(f"  {i}. {name}")
print(f"  ... and {len(feature_names) - 20} more features")

In [0]:
# Split data into training and testing sets
# Using 80/20 split with stratification to maintain class balance
X_train, X_test, y_train, y_test = train_test_split(
    X_transformed, 
    y, 
    test_size=0.2,      # 20% for testing
    random_state=42,     # Reproducible split
    stratify=y           # Maintain same class distribution in train and test
)

print(f"✓ Data split complete!")
print(f"\nTraining set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X_transformed)*100:.1f}%)")
print(f"Testing set:  {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X_transformed)*100:.1f}%)")
print(f"\nClass distribution in training set:")
print(pd.Series(y_train).value_counts().sort_index())
print("  0 = draw, 1 = home win, 2 = away win")

#Train Model Building
---------------------

In [0]:
# Train a Logistic Regression classifier
# multi_class='multinomial' handles 3 classes (home/draw/away)
# max_iter=1000 ensures convergence for the large feature set
print("Training Logistic Regression model...\n")

lr_model = LogisticRegression(
    multi_class='multinomial',  # Multinomial for 3-class classification
    solver='lbfgs',             # Good for multiclass problems
    max_iter=1000,              # Increase iterations for convergence
    random_state=42
)

# Fit the model
lr_model.fit(X_train, y_train)

# Make predictions on both train and test sets
y_train_pred = lr_model.predict(X_train)
y_test_pred = lr_model.predict(X_test)

# Calculate accuracies
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"✓ Model training complete!\n")
print(f"Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"Testing Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# Check for overfitting
overfit_gap = train_accuracy - test_accuracy
if overfit_gap > 0.05:
    print(f"\n⚠️  Warning: Possible overfitting (gap: {overfit_gap*100:.2f}%)")
else:
    print(f"\n✓ Good generalization (gap: {overfit_gap*100:.2f}%)")

#Model Evaluation
-------------------

In [0]:
# Calculate detailed classification metrics
from sklearn.metrics import precision_recall_fscore_support

# Get confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

print("="*60)
print("CONFUSION MATRIX (Test Set)")
print("="*60)
print("\nActual vs Predicted | Draw | Home | Away")
print("-" * 45)
for i, label in enumerate(['Draw', 'Home Win', 'Away Win']):
    print(f"{label:14} | {cm[i][0]:4d} | {cm[i][1]:4d} | {cm[i][2]:4d}")

# Calculate per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(y_test, y_test_pred, average=None)

# Calculate True Positives, False Positives, False Negatives, True Negatives for each class
metrics_dict = {}
for i, class_name in enumerate(['draw', 'home_win', 'away_win']):
    # For multiclass, we treat each class as binary (one-vs-all)
    tp = cm[i, i]  # True Positives: correctly predicted as this class
    fp = cm[:, i].sum() - tp  # False Positives: predicted as this class but actually another
    fn = cm[i, :].sum() - tp  # False Negatives: actually this class but predicted as another
    tn = cm.sum() - tp - fp - fn  # True Negatives: correctly predicted as NOT this class
    
    metrics_dict[class_name] = {
        'true_positives': int(tp),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_negatives': int(tn),
        'precision': float(precision[i]),
        'recall': float(recall[i]),
        'f1_score': float(f1[i]),
        'support': int(support[i])
    }

# Overall metrics
metrics_dict['overall'] = {
    'accuracy': float(test_accuracy),
    'train_accuracy': float(train_accuracy),
    'macro_precision': float(precision.mean()),
    'macro_recall': float(recall.mean()),
    'macro_f1': float(f1.mean()),
    'total_samples': len(y_test)
}

print("\n" + "="*60)
print("DETAILED METRICS BY CLASS")
print("="*60)
for class_name, metrics in metrics_dict.items():
    if class_name != 'overall':
        print(f"\n{class_name.upper().replace('_', ' ')}:")
        print(f"  TP: {metrics['true_positives']:4d}  |  FP: {metrics['false_positives']:4d}")
        print(f"  FN: {metrics['false_negatives']:4d}  |  TN: {metrics['true_negatives']:4d}")
        print(f"  Precision: {metrics['precision']:.4f}  |  Recall: {metrics['recall']:.4f}  |  F1: {metrics['f1_score']:.4f}")

print("\n" + "="*60)
print("OVERALL METRICS")
print("="*60)
print(f"Test Accuracy:  {metrics_dict['overall']['accuracy']:.4f} ({metrics_dict['overall']['accuracy']*100:.2f}%)")
print(f"Train Accuracy: {metrics_dict['overall']['train_accuracy']:.4f} ({metrics_dict['overall']['train_accuracy']*100:.2f}%)")
print(f"Macro Precision: {metrics_dict['overall']['macro_precision']:.4f}")
print(f"Macro Recall:    {metrics_dict['overall']['macro_recall']:.4f}")
print(f"Macro F1 Score:  {metrics_dict['overall']['macro_f1']:.4f}")

#Mlflow Set-up
------------------

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# Set experiment name
experiment_name = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/football_game_classification"
mlflow.set_experiment(experiment_name)

print(f"✓ MLflow experiment set: {experiment_name}")

In [0]:
# Start MLflow run and log the model
with mlflow.start_run(run_name="logistic_regression_sklearn") as run:
    
    # Log parameters
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("solver", "lbfgs")
    mlflow.log_param("max_iter", 1000)
    mlflow.log_param("multi_class", "multinomial")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("num_features", X_transformed.shape[1])
    mlflow.log_param("num_samples", len(y))
    
    # Log overall metrics
    mlflow.log_metric("test_accuracy", metrics_dict['overall']['accuracy'])
    mlflow.log_metric("train_accuracy", metrics_dict['overall']['train_accuracy'])
    mlflow.log_metric("macro_precision", metrics_dict['overall']['macro_precision'])
    mlflow.log_metric("macro_recall", metrics_dict['overall']['macro_recall'])
    mlflow.log_metric("macro_f1_score", metrics_dict['overall']['macro_f1'])
    
    # Log per-class metrics
    for class_name, metrics in metrics_dict.items():
        if class_name != 'overall':
            mlflow.log_metric(f"{class_name}_tp", metrics['true_positives'])
            mlflow.log_metric(f"{class_name}_fp", metrics['false_positives'])
            mlflow.log_metric(f"{class_name}_fn", metrics['false_negatives'])
            mlflow.log_metric(f"{class_name}_tn", metrics['true_negatives'])
            mlflow.log_metric(f"{class_name}_precision", metrics['precision'])
            mlflow.log_metric(f"{class_name}_recall", metrics['recall'])
            mlflow.log_metric(f"{class_name}_f1_score", metrics['f1_score'])
    
    # Create a pipeline with preprocessor and model for deployment
    from sklearn.pipeline import Pipeline as SklearnPipeline
    full_pipeline = SklearnPipeline([
        ('preprocessor', preprocessor),
        ('classifier', lr_model)
    ])
    
    # Infer signature from training data
    signature = infer_signature(X, lr_model.predict(X_transformed))
    
    # Log the complete pipeline (preprocessor + model)
    mlflow.sklearn.log_model(
        sk_model=full_pipeline,
        artifact_path="model",
        signature=signature,
        input_example=X.iloc[:5],
        registered_model_name=None  # We'll register separately to Unity Catalog
    )
    
    run_id = run.info.run_id
    
print(f"\n✓ Model and metrics logged successfully!")
print(f"Run ID: {run_id}")
print(f"Experiment: {experiment_name}")

# Register Model to Unity Catalog
----------------------------------

In [0]:
# Register the model to Unity Catalog
model_name = f"{catalog}.{gold_schema}.football_game_classifier"

# Register the model from the run
model_uri = f"runs:/{run_id}/model"
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name
)

print(f"\n✓ Model registered to Unity Catalog!")
print(f"Model Name: {model_name}")
print(f"Version: {registered_model.version}")
print(f"\nYou can view the model in Unity Catalog or use it for serving.")

#Create model Endpoint
--------------------------

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

# Initialize Databricks client
w = WorkspaceClient()

# Define endpoint name
endpoint_name = "football-game-classifier-endpoint"

# Check if endpoint already exists
try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists. Updating...")
    
    # Update existing endpoint with new model version
    w.serving_endpoints.update_config(
        name=endpoint_name,
        served_entities=[
            ServedEntityInput(
                entity_name=model_name,
                entity_version=registered_model.version,
                scale_to_zero_enabled=True,
                workload_size="Small"
            )
        ]
    )
    print(f"✓ Endpoint updated successfully!")
    
except Exception as e:
    if "RESOURCE_DOES_NOT_EXIST" in str(e) or "does not exist" in str(e).lower():
        print(f"Creating new endpoint '{endpoint_name}'...")
        
        # Create new endpoint
        w.serving_endpoints.create(
            name=endpoint_name,
            config=EndpointCoreConfigInput(
                served_entities=[
                    ServedEntityInput(
                        entity_name=model_name,
                        entity_version=registered_model.version,
                        scale_to_zero_enabled=True,
                        workload_size="Small"
                    )
                ]
            )
        )
        print(f"✓ Endpoint created successfully!")
    else:
        print(f"⚠️ Error: {e}")
        raise

print(f"\n" + "="*60)
print(f"MODEL SERVING ENDPOINT DETAILS")
print("="*60)
print(f"Endpoint Name: {endpoint_name}")
print(f"Model: {model_name}")
print(f"Version: {registered_model.version}")
print(f"\nThe endpoint is being deployed. This may take a few minutes.")
print(f"You can check the status in the Serving UI or wait for it to become ready.")

# Predictions
-----------------------------------
Now let's use the trained model to predict outcomes for future games (games that haven't been played yet).

Future games are identified by:
- **Null scores** (`home_score` or `away_score` is null)
- **Future dates** (date > today)

In [0]:
# Check if we have future games to predict on
future_count = df_future.count()

if future_count > 0:
    print(f"Preparing {future_count:,} future games for prediction...\n")
    
    # Convert future games to pandas
    df_future_pandas_full = df_future.toPandas()
    
    # Store game info for later display (id, date, teams, tournament, etc.)
    future_game_info = df_future_pandas_full[['id', 'date', 'home_team', 'away_team', 'tournament', 'city', 'country']].copy()
    
    # Prepare features - drop same columns as training (only those that exist)
    cols_to_drop = ['id', 'date', 'home_score', 'away_score', 'winner']
    # Add label column only if it exists (it won't for future games)
    if 'label' in df_future_pandas_full.columns:
        cols_to_drop.append('label')
    if 'home_or_away_winner' in df_future_pandas_full.columns:
        cols_to_drop.append('home_or_away_winner')
    
    X_future = df_future_pandas_full.drop(columns=cols_to_drop)
    
    # Ensure columns match training data exactly
    missing_cols = set(X.columns) - set(X_future.columns)
    extra_cols = set(X_future.columns) - set(X.columns)
    
    if missing_cols:
        print(f"⚠️  Adding missing columns: {missing_cols}")
        for col in missing_cols:
            X_future[col] = None
    
    if extra_cols:
        print(f"⚠️  Dropping extra columns: {extra_cols}")
        X_future = X_future.drop(columns=list(extra_cols))
    
    # Reorder to match training data
    X_future = X_future[X.columns]
    
    print(f"✓ Future games prepared!")
    print(f"  Shape: {X_future.shape}")
    print(f"  Columns match training: {list(X_future.columns) == list(X.columns)}")
    print(f"\nSample of games to predict:")
    display(future_game_info.head(10))
    
else:
    print("⚠️ No future games available for prediction.")
    print("All games in the dataset have already been played.")
    X_future = None
    future_game_info = None

In [0]:
if future_count > 0 and X_future is not None:
    print("Making predictions on future games...\n")
    
    # Use the full pipeline (preprocessor + model) to make predictions
    future_predictions = full_pipeline.predict(X_future)
    future_probabilities = full_pipeline.predict_proba(X_future)
    
    # Create results dataframe starting with game info
    results_df = future_game_info.copy()
    
    # Add predictions
    results_df['predicted_label'] = future_predictions
    
    # Map predictions to readable labels
    outcome_map = {0: 'Draw', 1: 'Home Win', 2: 'Away Win'}
    results_df['predicted_outcome'] = results_df['predicted_label'].map(outcome_map)
    
    # Add probability columns
    results_df['prob_draw'] = future_probabilities[:, 0]
    results_df['prob_home_win'] = future_probabilities[:, 1]
    results_df['prob_away_win'] = future_probabilities[:, 2]
    
    # Add confidence (max probability)
    results_df['confidence'] = future_probabilities.max(axis=1)
    
    print(f"✓ Predictions complete!\n")
    
    # Show prediction distribution
    print("="*60)
    print("PREDICTION SUMMARY")
    print("="*60)
    print(f"\nTotal games predicted: {len(results_df):,}")
    print(f"\nPrediction Distribution:")
    print(results_df['predicted_outcome'].value_counts())
    print(f"\nConfidence Statistics:")
    print(f"  Average: {results_df['confidence'].mean():.2%}")
    print(f"  Median:  {results_df['confidence'].median():.2%}")
    print(f"  Min:     {results_df['confidence'].min():.2%}")
    print(f"  Max:     {results_df['confidence'].max():.2%}")
    
else:
    print("Skipping predictions - no future games available.")
    results_df = None

In [0]:
if future_count > 0 and results_df is not None:
    print("="*100)
    print("FUTURE GAME PREDICTIONS")
    print("="*100)
    
    # Select key columns for display
    display_cols = [
        'home_team', 'away_team', 'tournament', 'city', 'country',
        'predicted_outcome', 'confidence',
        'prob_draw', 'prob_home_win', 'prob_away_win'
    ]
    
    # Filter to only include columns that exist
    display_cols = [col for col in display_cols if col in results_df.columns]
    
    predictions_display = results_df[display_cols].copy()
    
    # Format probability columns as percentages
    for col in ['prob_draw', 'prob_home_win', 'prob_away_win', 'confidence']:
        if col in predictions_display.columns:
            predictions_display[col] = predictions_display[col].apply(lambda x: f"{x:.1%}")
    
    # Sort by confidence (descending)
    if 'confidence' in predictions_display.columns:
        # Convert back to float for sorting
        predictions_display['confidence_sort'] = results_df['confidence']
        predictions_display = predictions_display.sort_values('confidence_sort', ascending=False)
        predictions_display = predictions_display.drop('confidence_sort', axis=1)
    
    print(f"\nTop {min(20, len(predictions_display))} predictions (sorted by confidence):\n")
    display(predictions_display.head(20))
    
    # Show high confidence predictions
    high_confidence = results_df[results_df['confidence'] > 0.7]
    if len(high_confidence) > 0:
        print(f"\n\nHigh Confidence Predictions (>70%): {len(high_confidence):,} games")
        print("="*100)
        display(predictions_display[predictions_display.index.isin(high_confidence.index)].head(10))
    
else:
    print("No future games to display.")

In [0]:
if future_count > 0 and results_df is not None:
    print("Saving predictions to Delta table...\n")
    
    # Convert predictions back to Spark DataFrame
    predictions_spark = spark.createDataFrame(results_df)
    
    # Define table name for predictions
    predictions_table = f"{catalog}.{gold_schema}.future_game_predictions"
    
    # Write to Delta table (overwrite mode to replace previous predictions)
    predictions_spark.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(predictions_table)
    
    print(f"✓ Predictions saved successfully!")
    print(f"\nTable: {predictions_table}")
    print(f"Rows: {len(results_df):,}")
    print(f"\nColumns saved:")
    for col in results_df.columns:
        print(f"  - {col}")
    print(f"\nYou can query the predictions using:")
    print(f"  SELECT * FROM {predictions_table}")
    print(f"  WHERE confidence > 0.7  -- High confidence predictions only")
    
else:
    print("No predictions to save.")

#Evaluation of Predictions
------------------------------

In [0]:
from pyspark.sql.functions import when

display(
    predictions_spark.withColumn(
        'Winner_Country',
        when(predictions_spark['predicted_outcome'] == 'Home Win', predictions_spark['home_team'])
        .when(predictions_spark['predicted_outcome'] == 'Away Win', predictions_spark['away_team'])
        .otherwise('Draw')
    )
)